In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = "my_clean_3_with_weather.parquet"
TARGET_COL = "total_amount"


In [3]:
print("Loading latest dataset...")
df = pd.read_parquet(DATA_PATH)

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")

# Keep only 2023-2024, just in case the file contains date outliers.
df = df.loc[
    df["tpep_pickup_datetime"].ge(pd.Timestamp("2023-01-01"))
    & df["tpep_pickup_datetime"].lt(pd.Timestamp("2025-01-01"))
].copy()

print("Shape:", df.shape)
print("Date range:", df["tpep_pickup_datetime"].min(), "-", df["tpep_pickup_datetime"].max())
print("Trips by year:")
print(df["tpep_pickup_datetime"].dt.year.value_counts().sort_index())
df.head()


Loading latest dataset...
Shape: (7655312, 34)
Date range: 2023-01-01 00:01:58 - 2024-12-31 23:53:03
Trips by year:
tpep_pickup_datetime
2023    3681755
2024    3973557
Name: count, dtype: int64


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,DO_Borough,DO_Zone,DO_lon,DO_lat,distance_group,duration_min,temperature,precipitation,snowfall,weather_code
0,2,2023-01-29 17:52:02,2023-01-29 17:56:43,1,1.17,1.0,N,262,74,2,...,Manhattan,East Harlem North,-73.937346,40.801169,short,4.683333,9.6,0.0,0.0,3
1,1,2023-01-08 15:57:24,2023-01-08 16:02:47,1,0.90,1.0,N,229,237,2,...,Manhattan,Upper East Side South,-73.965635,40.768615,very_short,5.383333,4.2,0.0,0.0,3
2,2,2023-01-21 19:38:01,2023-01-21 19:45:02,1,0.95,1.0,N,45,261,1,...,Manhattan,World Trade Center,-74.013023,40.709139,very_short,7.016667,3.1,0.0,0.0,3
3,2,2023-01-23 16:07:31,2023-01-23 16:26:46,5,0.88,1.0,N,237,141,1,...,Manhattan,Lenox Hill West,-73.959635,40.766948,very_short,19.250000,2.9,0.6,0.0,53
4,2,2023-01-26 21:21:08,2023-01-26 21:24:48,2,1.03,1.0,N,229,140,1,...,Manhattan,Lenox Hill East,-73.954739,40.765484,short,3.666667,5.0,0.0,0.0,1


In [ ]:
print("1. Building time, business, geo and weather features...")

model_df = df.copy()

# Fill low-cardinality categorical columns before splitting/features.
model_df["store_and_fwd_flag"] = model_df["store_and_fwd_flag"].fillna("unknown").astype(str)
model_df["distance_group"] = model_df["distance_group"].astype(str).replace("nan", "unknown")
model_df["PU_Borough"] = model_df["PU_Borough"].fillna("unknown").astype(str)
model_df["DO_Borough"] = model_df["DO_Borough"].fillna("unknown").astype(str)

model_df["pickup_hour"] = model_df["tpep_pickup_datetime"].dt.hour
model_df["pickup_dayofweek"] = model_df["tpep_pickup_datetime"].dt.dayofweek
model_df["pickup_month"] = model_df["tpep_pickup_datetime"].dt.month
model_df["pickup_day"] = model_df["tpep_pickup_datetime"].dt.day
model_df["pickup_weekofyear"] = model_df["tpep_pickup_datetime"].dt.isocalendar().week.astype(int)
model_df["is_weekend"] = model_df["pickup_dayofweek"].isin([5, 6]).astype(int)
model_df["is_rush_hour"] = (
    (model_df["pickup_dayofweek"] < 5)
    & (model_df["pickup_hour"] >= 16)
    & (model_df["pickup_hour"] < 20)
).astype(int)
model_df["is_night_tariff"] = ((model_df["pickup_hour"] >= 20) | (model_df["pickup_hour"] < 6)).astype(int)

model_df["hour_sin"] = np.sin(2 * np.pi * model_df["pickup_hour"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["pickup_hour"] / 24)
model_df["dow_sin"] = np.sin(2 * np.pi * model_df["pickup_dayofweek"] / 7)
model_df["dow_cos"] = np.cos(2 * np.pi * model_df["pickup_dayofweek"] / 7)
model_df["month_sin"] = np.sin(2 * np.pi * model_df["pickup_month"] / 12)
model_df["month_cos"] = np.cos(2 * np.pi * model_df["pickup_month"] / 12)

model_df["lat_diff"] = (model_df["DO_lat"] - model_df["PU_lat"]).abs()
model_df["lon_diff"] = (model_df["DO_lon"] - model_df["PU_lon"]).abs()
model_df["gps_distance"] = model_df["lat_diff"] + model_df["lon_diff"]
model_df["distance_ratio"] = model_df["trip_distance"] / (model_df["gps_distance"] + 0.001)
model_df["distance_ratio"] = model_df["distance_ratio"].clip(0, 15)

model_df["duration_min"] = model_df["duration_min"].clip(lower=0.1)
model_df["speed_mph"] = model_df["trip_distance"] / (model_df["duration_min"] / 60 + 0.01)
model_df["speed_mph"] = model_df["speed_mph"].clip(0, 80)
model_df["log_trip_distance"] = np.log1p(model_df["trip_distance"])
model_df["log_duration_min"] = np.log1p(model_df["duration_min"])

model_df["has_precipitation"] = (model_df["precipitation"] > 0).astype(int)
model_df["has_snowfall"] = (model_df["snowfall"] > 0).astype(int)
model_df["bad_weather"] = ((model_df["precipitation"] > 0) | (model_df["snowfall"] > 0)).astype(int)
model_df["precipitation_x_rush_hour"] = model_df["precipitation"] * model_df["is_rush_hour"]
model_df["snowfall_x_rush_hour"] = model_df["snowfall"] * model_df["is_rush_hour"]

airport_zone_ids = {1, 132, 138}
model_df["pickup_airport"] = model_df["PULocationID"].isin(airport_zone_ids).astype(int)
model_df["dropoff_airport"] = model_df["DOLocationID"].isin(airport_zone_ids).astype(int)
model_df["airport_trip"] = ((model_df["pickup_airport"] == 1) | (model_df["dropoff_airport"] == 1)).astype(int)
model_df["same_zone"] = (model_df["PULocationID"] == model_df["DOLocationID"]).astype(int)
model_df["same_borough"] = (model_df["PU_Borough"] == model_df["DO_Borough"]).astype(int)
model_df["interborough_trip"] = (model_df["PU_Borough"] != model_df["DO_Borough"]).astype(int)

model_df["route_id"] = model_df["PULocationID"].astype(str) + "_" + model_df["DOLocationID"].astype(str)

print("Features created")


In [ ]:
categorical_cols = [
    "VendorID",
    "RatecodeID",
    "payment_type",
    "store_and_fwd_flag",
    "PU_Borough",
    "DO_Borough",
    "distance_group",
]

target_encoding_cols = [
    "PULocationID",
    "DOLocationID",
    "route_id",
    "RatecodeID",
]

target_encoded_feature_names = [
    "PU_zone_average_price",
    "DO_zone_average_price",
    "route_average_price",
    "ratecode_average_price",
]

weather_cols = [
    "temperature",
    "precipitation",
    "snowfall",
    "weather_code",
    "has_precipitation",
    "has_snowfall",
    "bad_weather",
    "precipitation_x_rush_hour",
    "snowfall_x_rush_hour",
]

time_cols = [
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "pickup_day",
    "pickup_weekofyear",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

trip_cols = [
    "trip_distance",
    "passenger_count",
    "is_weekend",
    "is_rush_hour",
    "is_night_tariff",
    "gps_distance",
    "distance_ratio",
    "duration_min",
    "speed_mph",
    "log_trip_distance",
    "log_duration_min",
    "pickup_airport",
    "dropoff_airport",
    "airport_trip",
    "same_zone",
    "same_borough",
    "interborough_trip",
]

numerical_cols = [
    *trip_cols,
    *time_cols,
    *weather_cols,
    *target_encoded_feature_names,
]

base_feature_cols = [
    "PULocationID",
    "DOLocationID",
    "route_id",
    "tpep_pickup_datetime",
    *categorical_cols,
    *[col for col in numerical_cols if col not in target_encoded_feature_names],
]

model_df = model_df.dropna(subset=[TARGET_COL, *base_feature_cols]).copy()
model_df = model_df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

# Time split: train on earlier data, test on the latest months.
test_start = pd.Timestamp("2024-10-01")
train_mask = model_df["tpep_pickup_datetime"] < test_start
test_mask = model_df["tpep_pickup_datetime"] >= test_start

if test_mask.sum() == 0 or train_mask.sum() == 0:
    split_idx = int(len(model_df) * 0.8)
    train_mask = model_df.index < split_idx
    test_mask = model_df.index >= split_idx
    print("Fallback split: first 80% train, last 20% test")
else:
    print("Time split: train before", test_start.date(), "test from", test_start.date())

X_train_base = model_df.loc[train_mask, base_feature_cols].copy()
X_test_base = model_df.loc[test_mask, base_feature_cols].copy()
y_train = model_df.loc[train_mask, TARGET_COL].copy()
y_test = model_df.loc[test_mask, TARGET_COL].copy()

X_train_base = X_train_base.drop(columns=["tpep_pickup_datetime"])
X_test_base = X_test_base.drop(columns=["tpep_pickup_datetime"])

print("Train shape:", X_train_base.shape)
print("Test shape:", X_test_base.shape)
print("Train target mean:", round(y_train.mean(), 2))
print("Test target mean:", round(y_test.mean(), 2))


In [ ]:
def add_target_encoding(train_x, test_x, train_y, source_cols, encoded_cols, smooth=50):
    train_x = train_x.copy()
    test_x = test_x.copy()
    global_mean = float(train_y.mean())

    train_with_target = train_x.copy()
    train_with_target[TARGET_COL] = train_y.values

    for source_col, encoded_col in zip(source_cols, encoded_cols):
        stats = train_with_target.groupby(source_col, observed=False)[TARGET_COL].agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smooth) / (stats["count"] + smooth)
        train_x[encoded_col] = train_x[source_col].map(smoothed).fillna(global_mean).astype(float)
        test_x[encoded_col] = test_x[source_col].map(smoothed).fillna(global_mean).astype(float)

    return train_x, test_x

print("2. Adding smoothed train-only target encoding...")
X_train_full, X_test_full = add_target_encoding(
    X_train_base,
    X_test_base,
    y_train,
    target_encoding_cols,
    target_encoded_feature_names,
)

X_train = X_train_full[numerical_cols + categorical_cols].copy()
X_test = X_test_full[numerical_cols + categorical_cols].copy()

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print("Final feature count:", X_train.shape[1])
print("Final train shape:", X_train.shape)
print("Final test shape:", X_test.shape)


In [7]:
print("3. Training HistGradientBoostingRegressor...")

boosting_model = HistGradientBoostingRegressor(
    categorical_features="from_dtype",
    max_iter=900,
    max_leaf_nodes=127,
    learning_rate=0.015,
    min_samples_leaf=40,
    l2_regularization=0.01,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42,
)

boosting_model.fit(X_train, y_train)
print("Boosting model trained")
print("Iterations used:", boosting_model.n_iter_)


3. Training HistGradientBoostingRegressor...
Boosting model trained
Iterations used: 900


In [8]:
print("4. Evaluating the boosting model...")

y_pred = boosting_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("================ BOOSTING METRICS ================")
print(f"MAE:  {mae:.2f} $")
print(f"RMSE: {rmse:.2f} $")
print(f"R2:   {r2 * 100:.1f}%")
print("==================================================")


4. Evaluating the boosting model...
================ BOOSTING METRICS ================
MAE:  1.84 $
RMSE: 3.80 $
R2:   96.8%


In [ ]:
prediction_preview = pd.DataFrame({
    "y_true": y_test.reset_index(drop=True).head(20),
    "y_pred": pd.Series(y_pred).head(20),
})

prediction_preview["abs_error"] = (prediction_preview["y_true"] - prediction_preview["y_pred"]).abs()
prediction_preview
